<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Clustering Jerárquico: El Árbol de los Parecidos 🌳
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 04
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/04%20-%20Clustering%20No%20Supervisado/Para%20Dummies/02_Jerarquico_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del [cuaderno 02 (Clustering Jerárquico)](../02_Clustering_Jerarquico.ipynb). En el cuaderno anterior elegíamos **de antemano** cuántos grupos formar. El método jerárquico hace algo distinto: **construye un árbol completo** que va de «cada uno por su lado» hasta «todos juntos», y **tú decides después** dónde cortarlo. Es como un **árbol genealógico al revés**.

Al terminar podrás explicar:
1. Cómo se construye el **árbol** juntando siempre a los más parecidos.
2. Cómo se **lee** un dendrograma y cómo se **corta** para obtener grupos.
3. Qué cambia según cómo midas la distancia **entre dos grupos** (cuatro formas).
4. Qué es el problema de la **cadena de amigos de amigos**.

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../02_Clustering_Jerarquico.ipynb). Otras miradas sencillas: [Clustering Jerárquico (Dummies) en Data Mining](../../../Data%20Mining/03%20-%20Clustering%20y%20Mineria%20Reglas%20de%20Asociacion/Para%20Dummies/01_Jerarquico_Dummies.ipynb) y [Dendrogramas (Dummies) en Data Science Programming](../../../Data%20Science%20programming/10%20-%20Clustering/Para%20Dummies/03_Clustering_Jerarquico_y_Dendrogramas_Dummies.ipynb).

---
## 1. Juntar amigos en grupos cada vez más grandes 🤝

Imagina una **fiesta** donde nadie conoce a nadie, y cada persona es, al principio, **un grupo de una sola persona**. El método hace esto, una y otra vez:

1. Busca **las dos personas (o grupos) más parecidas** entre todas.
2. **Júntalas** en un solo grupo.
3. Repite hasta que **todos estén en un único grupo gigante**.

Lo que queda registrado es **el orden de las uniones y qué tan parecidos eran al unirse**. Eso se dibuja como un árbol llamado **dendrograma**: las uniones **bajas** son entre cosas muy parecidas; las uniones **altas**, entre cosas muy distintas.

Probémoslo con algo familiar: **ciudades de Colombia** descritas solo por su ubicación (latitud y longitud aproximadas). ¿El árbol juntará a las ciudades vecinas?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Ubicacion APROXIMADA (latitud, longitud en grados); para este ejemplo las tratamos como un plano
ciudades = {"Bogotá": (4.71, -74.07), "Medellín": (6.25, -75.57), "Cali": (3.45, -76.53), "Barranquilla": (10.96, -74.80),
            "Cartagena": (10.39, -75.51), "Bucaramanga": (7.12, -73.12), "Cúcuta": (7.89, -72.51), "Pereira": (4.81, -75.69),
            "Manizales": (5.07, -75.52), "Santa Marta": (11.24, -74.20), "Villavicencio": (4.15, -73.63), "Neiva": (2.93, -75.28),
            "Pasto": (1.21, -77.28), "Ibagué": (4.44, -75.23), "Montería": (8.75, -75.88), "Tunja": (5.54, -73.36),
            "Valledupar": (10.48, -73.25), "Popayán": (2.44, -76.61)}
nombres = list(ciudades)
coords = np.array(list(ciudades.values()))
X = coords[:, ::-1]                                    # (longitud, latitud): x = este-oeste, y = norte-sur

Z = linkage(X, method="average")                       # la "bitacora" de uniones: quien se junto con quien y a que altura
corte = (Z[-4, 2] + Z[-3, 2]) / 2                      # altura de corte que deja exactamente 4 grupos (entre la 4.a y la 3.a ultima union)
plt.figure(figsize=(9, 6))
dendrogram(Z, labels=nombres, orientation="right", color_threshold=corte, leaf_font_size=10)
plt.axvline(corte, color="black", ls="--", label="corte (deja 4 grupos)")
plt.xlabel("Qué tan distintas eran las ciudades cuando se juntaron (altura de la unión)")
plt.title("Árbol de ciudades colombianas por su ubicación", fontweight="bold"); plt.legend(); plt.show()
print("Fíjate: primero se juntan las ciudades vecinas (uniones bajas) y al final las regiones lejanas (uniones altas).")

---
## 2. Cortar el árbol ✂️

El árbol completo no es un agrupamiento todavía: **tú decides dónde cortarlo**.

- Si cortas **abajo**, quedan **muchos grupos pequeños**.
- Si cortas **arriba**, quedan **pocos grupos grandes**.
- Una buena pista: corta donde hay **una rama vertical muy larga** (un «salto» grande), porque ahí se unieron cosas **muy distintas**.

También puedes decir «quiero exactamente **4 grupos**» y el programa corta a la altura que corresponda.

In [ ]:
grupos = fcluster(Z, 4, criterion="maxclust")                       # cortar para obtener exactamente 4 grupos
tabla = pd.DataFrame({"ciudad": nombres, "grupo": grupos}).sort_values(["grupo", "ciudad"])
for g, sub in tabla.groupby("grupo"):
    print(f"Grupo {g}: " + ", ".join(sub["ciudad"]))

plt.figure(figsize=(6.5, 6.5))
plt.scatter(X[:, 0], X[:, 1], c=grupos, cmap="tab10", s=140, edgecolor="black")
for (x, y), nombre in zip(X, nombres):
    plt.annotate(nombre, (x, y), textcoords="offset points", xytext=(6, 5), fontsize=8)
plt.xlabel("Longitud (oeste a este)"); plt.ylabel("Latitud (sur a norte)"); plt.title("Cuatro grupos de ciudades al cortar el árbol", fontweight="bold")
plt.show()

---
## 3. ¿Cómo se mide la distancia entre DOS GRUPOS? 📏

Medir la distancia entre dos **personas** es fácil. Pero cuando ya hay **grupos**, ¿cuál es la distancia entre el grupo A y el grupo B? Imagina **dos pueblos** y quieres saber qué tan lejos están:

| Forma de medir | En palabras | Efecto |
|---|---|---|
| **Single** (el vecino más cercano) | La distancia entre **las dos casas más cercanas** de cada pueblo | Une pueblos si **una sola casa** está cerca de la otra: forma grupos **alargados**, pero se deja **engañar por cadenas** |
| **Complete** (el vecino más lejano) | La distancia entre **las dos casas más lejanas** | Solo une si **todo** el pueblo está cerca: grupos **compactos** |
| **Average** (promedio) | El **promedio** de todas las distancias entre casas | Algo intermedio |
| **Ward** | Une los grupos que **menos aumentan** lo «dispersos» que queda el grupo resultante | Grupos **redondos y de tamaño parecido** (suele ser una buena opción por defecto) |

Veamos cómo cambian los **4 grupos** de ciudades según la forma de medir.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4.6))
for ax, metodo in zip(axes, ["single", "complete", "average", "ward"]):
    g = fcluster(linkage(X, method=metodo), 4, criterion="maxclust")
    ax.scatter(X[:, 0], X[:, 1], c=g, cmap="tab10", s=110, edgecolor="black")
    ax.set_title(f"{metodo}\ntamaños de grupo: {[int(v) for v in sorted(np.bincount(g)[1:], reverse=True)]}", fontweight="bold", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Los mismos puntos, cuatro formas de medir la distancia entre grupos", fontweight="bold")
plt.tight_layout(); plt.show()
print("Con 'single' suelen quedar grupos muy desiguales (uno enorme y otros diminutos); 'complete' y 'ward' reparten de forma más pareja.")

---
## 4. El problema de la cadena de amigos de amigos ⛓️

El método *single* (vecino más cercano) tiene un defecto famoso. Piensa en **dos barrios** separados por un descampado, pero con **una hilera de casas** que los conecta. Para *single*, basta con que **cada casa esté cerca de la siguiente** para juntar todo en un solo grupo: es la **cadena de «amigos de amigos»**. Unas pocas casas «puente» arruinan una separación que era evidente.

Lo vemos con **dos grupos claros** al que añadimos 12 puntos «puente». Pedimos 2 grupos con *single* y con *complete*.

In [ ]:
rng = np.random.RandomState(42)
barrio_a = rng.normal(loc=[-3, 0], scale=0.7, size=(100, 2))
barrio_b = rng.normal(loc=[3, 0], scale=0.7, size=(100, 2))
puente = np.c_[np.linspace(-2, 2, 14)[1:-1], rng.normal(scale=0.08, size=12)]                # 12 casas en hilera entre los barrios
todo = np.vstack([barrio_a, barrio_b, puente])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for ax, metodo in zip(axes, ["single", "complete"]):
    g = fcluster(linkage(todo, method=metodo), 2, criterion="maxclust")
    ax.scatter(todo[:, 0], todo[:, 1], c=g, cmap="coolwarm", s=16)
    ax.set_title(f"{metodo}: tamaños de los 2 grupos = {[int(v) for v in sorted(np.bincount(g)[1:], reverse=True)]}", fontweight="bold", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("'single' dejó casi todo en un solo grupo (la cadena de casas unió los dos barrios); 'complete' separó los dos barrios como esperábamos.")

---
##### 🛠️ Práctica 1: ¿Con quién se junta Tunja? Cortando el árbol a distintas alturas

Volvamos al árbol de ciudades colombianas (`Z`, `nombres`). Vamos a ver cómo cambia **la compañía de Tunja** según dónde cortes el árbol:

1. Corta el árbol en **2**, **3** y **4** grupos con `fcluster(Z, K, criterion="maxclust")` y guarda las etiquetas en un diccionario `cortes` (clave = número de grupos).
2. Para cada corte, saca la lista de ciudades que están en **el mismo grupo que Tunja** y guárdala en `con_tunja[K]`.
3. Imprime cuántas ciudades tiene el grupo de Tunja en cada corte.
4. Comprueba que con 4 grupos el grupo de Tunja tiene 7 ciudades e incluye a Bogotá, y que **al cortar más abajo el grupo de Tunja nunca crece**.

In [ ]:
# Escribe tu código aquí

# 1) Tres cortes del mismo arbol
# cortes = {K: fcluster(Z, K, criterion="maxclust") for K in (2, 3, 4)}

# 2) Ciudades que estan en el mismo grupo que Tunja
# posicion_tunja = nombres.index("Tunja")
# con_tunja = {}
# for K, etiquetas in cortes.items():
#     mismo_grupo = etiquetas == etiquetas[posicion_tunja]
#     con_tunja[K] = [c for c, m in zip(nombres, mismo_grupo) if m]

# 3) Cuantas ciudades acompanan a Tunja en cada corte
# for K in cortes:
#     print(...)

# 4) Comprobaciones
# assert len(con_tunja[4]) == 7 and "Bogotá" in con_tunja[4]
# assert len(con_tunja[2]) >= len(con_tunja[3]) >= len(con_tunja[4])


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# 1) Tres cortes del mismo arbol
cortes = {K: fcluster(Z, K, criterion="maxclust") for K in (2, 3, 4)}

# 2) Ciudades que estan en el mismo grupo que Tunja
posicion_tunja = nombres.index("Tunja")
con_tunja = {}
for K, etiquetas in cortes.items():
    mismo_grupo = etiquetas == etiquetas[posicion_tunja]
    con_tunja[K] = [c for c, m in zip(nombres, mismo_grupo) if m]

# 3) Cuantas ciudades acompanan a Tunja en cada corte
for K in cortes:
    print(f"Con {K} grupos, Tunja va con {len(con_tunja[K])} ciudades: {', '.join(con_tunja[K])}")

# 4) Comprobaciones
assert len(con_tunja[4]) == 7 and "Bogotá" in con_tunja[4]
assert len(con_tunja[2]) >= len(con_tunja[3]) >= len(con_tunja[4])
print("\nCortar mas abajo solo parte grupos: el grupo de Tunja se achica o se queda igual, nunca crece.")
```
</details>

---


---
## Resumen en una frase 🎯

> **El método jerárquico va juntando siempre a los dos grupos más parecidos hasta formar un solo árbol (el dendrograma); tú decides dónde cortarlo para obtener los grupos, y el resultado cambia según cómo midas la distancia entre grupos: el vecino más cercano puede ser engañado por cadenas de puntos, mientras que otras formas (completo, promedio, Ward) son más robustas.**

### Lo que aprendiste hoy:
- ✅ Se parte de **cada uno solo** y se **junta siempre lo más parecido** hasta quedar uno.
- ✅ El **dendrograma** registra las uniones: **bajo = muy parecidos**, **alto = muy distintos**.
- ✅ **No hay que decidir de antemano** cuántos grupos: se **corta** el árbol (por altura o por número).
- ✅ Hay **cuatro formas** de medir la distancia entre grupos; **Ward** suele ser una buena opción por defecto.
- ✅ El vecino más cercano (*single*) cae en la **trampa de la cadena** de puntos puente.

> 🎓 **Siguiente paso:** [DBSCAN (Dummies)](03_DBSCAN_Dummies.ipynb). Versión con más detalle de este tema: [cuaderno estándar](../02_Clustering_Jerarquico.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿qué significa que dos ramas del dendrograma se unan a una **altura muy alta**? ¿Cómo usarías eso para decidir dónde cortar?
2. Tienes un mapa de tiendas y hay una carretera con muchas tiendas pequeñas que conecta dos ciudades. ¿Qué forma de medir la distancia entre grupos te podría dar un solo grupo gigante por culpa de esa carretera, y por qué?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>